# Week 4.2: Modules and packages — Teacher notebook

**Exercises and solutions only.** The theory is in the student notebook, and exercise numbers are the same in both.

* Code cells contain the **solution**, followed by the automatic check that students also get.
  Cells with no `TODO` in the student version (setup cells, code to run and observe) appear here unchanged.
* The setup cell below recreates what the theory part builds (`fibo.py` and the toy `sound` package),
  so this notebook can be run from top to bottom on its own.
* The exercises create small scratch files (`mymath.py`, `greeter.py`, `toolbox/`, `textkit/`, ...); the last cell removes them.

| Part | Exercises |
|---|---|
| 1. Scopes and namespaces | 1.1 – 1.5 |
| 2. Modules | 2.1 – 2.8 |
| 3. Packages | 3.1 – 3.7 |

### Setup

Recreates `fibo.py` and the `sound` package exactly as built in the theory notebook.

In [ ]:
fibo_source = '''
# Fibonacci numbers module

def fib(n):    # write Fibonacci series up to n
    a, b = 0, 1
    while a < n:
        print(a, end=' ')
        a, b = b, a + b
    print()

def fib2(n):   # return Fibonacci series up to n
    result = []
    a, b = 0, 1
    while a < n:
        result.append(a)
        a, b = b, a + b
    return result
'''

with open('fibo.py', 'w') as f:
    f.write(fibo_source)

print('fibo.py written to disk')

script_addition = '''

if __name__ == "__main__":
    import sys
    fib(int(sys.argv[1]) if len(sys.argv) > 1 else 100)
'''

with open('fibo.py', 'a') as f:
    f.write(script_addition)

print('Updated fibo.py:')
with open('fibo.py') as f:
    print(f.read())

import os

# Build the directory tree
os.makedirs('sound/formats', exist_ok=True)
os.makedirs('sound/effects', exist_ok=True)
os.makedirs('sound/filters', exist_ok=True)

files = {
    'sound/__init__.py': "'''The sound package: tools for handling sound files and data.'''\n",
    'sound/formats/__init__.py': "'''Subpackage for file format conversions.'''\n",
    'sound/formats/wavread.py': (
        'def read(path):\n'
        '    print(f"[wavread] pretending to read {path}")\n'
        '    return b"fake-wav-data"\n'
    ),
    'sound/formats/wavwrite.py': (
        'def write(path, data):\n'
        '    print(f"[wavwrite] pretending to write {len(data)} bytes to {path}")\n'
    ),
    'sound/effects/__init__.py': "'''Subpackage for sound effects.'''\n",
    'sound/effects/echo.py': (
        'def echofilter(data, delay=0.1, atten=0.4):\n'
        '    print(f"[echo] applying echo (delay={delay}, atten={atten}) to {data!r}")\n'
        '    return data\n'
    ),
    'sound/effects/surround.py': (
        'def surround(data, channels=5):\n'
        '    print(f"[surround] spreading {data!r} across {channels} channels")\n'
        '    return data\n'
    ),
    'sound/filters/__init__.py': "'''Subpackage for filters.'''\n",
    'sound/filters/equalizer.py': (
        'def equalize(data, preset="flat"):\n'
        '    print(f"[equalizer] applying {preset!r} preset to {data!r}")\n'
        '    return data\n'
    ),
}

for path, content in files.items():
    with open(path, 'w') as f:
        f.write(content)

print("Package tree created:")
for root, dirs, filenames in os.walk('sound'):
    level = root.replace('sound', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'sound'}/")
    for fn in filenames:
        print(f"{indent}  {fn}")

import importlib
importlib.invalidate_caches()
import fibo

## Part 1: Python scopes and namespace

#### Exercise 1.1 — Reading the LEGB rule

For each name *used* inside `inner` (`len`, `str`, `print`, `x`, `y`, `z`), decide in which scope Python finds it:
`'L'` (local), `'E'` (enclosing), `'G'` (global) or `'B'` (built-in).
Fill in the dictionary `scope_of` (run the cell first to see the code work).

In [ ]:
x = 10

def outer():
    y = 20

    def inner():
        z = 30
        print(len(str(x + y + z)))

    inner()

outer()

scope_of = {
    "len": "B",
    "str": "B",
    "print": "B",
    "x": "G",
    "y": "E",
    "z": "L",
}

# --- Check your answer (do not edit) ---
import hashlib, json
_digest = hashlib.sha256(json.dumps(scope_of, sort_keys=True).encode()).hexdigest()[:10]
assert _digest == "0ff8980416", "Not quite. Remember the order: Local, Enclosing, Global, Built-in."
print("Correct!")

**Teaching notes.** `z` is assigned in `inner` (L); `y` is assigned in `outer`, the enclosing function (E); `x` lives at module level (G); `print`, `len` and `str` are found last, in the `builtins` namespace (B).

#### Exercise 1.2 — The `global` statement

The function below tries to add to a module-level running total, but something is wrong.
Run the cell (the `try/except` just prints the error instead of stopping the notebook).

In [ ]:
running_total = 0

def broken_add(n):
    running_total += n

try:
    broken_add(5)
except Exception as e:
    print(type(e).__name__ + ":", e)

**(a)** Which error do you get? Why does Python talk about a *local* variable, even though the code never wrote `running_total = ...`?

**Answer:** It is an `UnboundLocalError`. `running_total += n` is an *assignment* (it means `running_total = running_total + n`). Whenever a name is assigned anywhere in a function body, Python treats it as **local for the whole body**. So the right-hand side looks for a local `running_total` that has no value yet.

**(b)** Write a corrected version, `add_to_total(n)`, that really updates the module-level `running_total`.

In [ ]:
running_total = 0

def add_to_total(n):
    global running_total
    running_total += n

add_to_total(5)
add_to_total(7)

# --- Check your answer (do not edit) ---
assert running_total == 12, f"expected 12, got {running_total}"
print("Correct!")

**(c)** The next function modifies a global list and works fine **without** any `global` statement.

In [ ]:
log = []

def record(msg):
    log.append(msg)

record("hello")
print(log)

Why is `global` not needed in `record`, while it was needed in `add_to_total`?

**Answer:** `log.append(msg)` only *reads* the name `log` (found through the LEGB lookup, in the global scope) and then mutates the list object it refers to; the name itself is never rebound. `global` is needed only when the function *rebinds* the name (`=`, `+=`, `del`, ...). Note that `log += [msg]` **would** need it.

#### Exercise 1.3 — The `nonlocal` statement

Write a function `make_accumulator(start=0)` that returns a function `add(n)`.
Each call `add(n)` adds `n` to a running total (initially `start`) and returns the new total.
Two accumulators created separately must not share their totals.

In [ ]:
def make_accumulator(start=0):
    total = start

    def add(n):
        nonlocal total
        total += n
        return total

    return add

# --- Check your answer (do not edit) ---
acc1 = make_accumulator()
acc2 = make_accumulator(100)
assert acc1(5) == 5
assert acc1(10) == 15
assert acc2(1) == 101
assert acc1(1) == 16, "accumulators must be independent"
print("Correct!")

**Teaching notes.** Without `nonlocal total`, `total += n` would raise an `UnboundLocalError` (same mechanism as in Exercise 1.2).

#### Exercise 1.4 — Three different `x`

**(a)** Predict what the following code prints, *then* run the cell to check your answer.

In [ ]:
x = "global"

def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer()
print("module:", x)

Write down your prediction and explain it in one or two sentences.

**Answer:** ```
inner: local
outer: enclosing
module: global
```
Each assignment `x = ...` creates a *new local* variable in the function where it appears, so there are three different variables called `x`, one per namespace.

**(b)** Add **one** statement at the marked place so that `inner` rebinds the `x` of the *enclosing* function `outer_b`.
Expected output:
```
inner: local
outer: local
module: global
```

In [ ]:
x = "global"

def outer_b():
    x = "enclosing"

    def inner():
        nonlocal x
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer_b()
print("module:", x)

# --- Check your answer (do not edit) ---
import io, contextlib
x = "global"
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    outer_b()
    print("module:", x)
assert buf.getvalue().splitlines() == ["inner: local", "outer: local", "module: global"], buf.getvalue()
print("Correct!")

**(c)** Now add **one** statement so that `inner` rebinds the *module-level* `x` instead. Expected output:
```
inner: local
outer: enclosing
module: local
```

In [ ]:
x = "global"

def outer_c():
    x = "enclosing"

    def inner():
        global x
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer_c()
print("module:", x)

# --- Check your answer (do not edit) ---
import io, contextlib
x = "global"
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    outer_c()
    print("module:", x)
assert buf.getvalue().splitlines() == ["inner: local", "outer: enclosing", "module: local"], buf.getvalue()
x = "global"   # restore
print("Correct!")

#### Exercise 1.5 — `dir()` and namespaces

**(a)** Store in `result_a` a boolean that is `True` when `'abs'` is one of the built-in names but is **not** among the names of the current namespace.

In [ ]:
import builtins

result_a = 'abs' in dir(builtins) and 'abs' not in dir()

# --- Check your answer (do not edit) ---
assert result_a is True
print("Correct!")

**(b)** Inside a function, `dir()` without argument lists the names of the *local* namespace. Complete `local_names` so that it returns them.

In [ ]:
def local_names(a, b):
    c = a + b
    return dir()

# --- Check your answer (do not edit) ---
assert local_names(1, 2) == ['a', 'b', 'c']
print("Correct!")

**(c)** Predict what happens at the first `print`, then finish the cell: add **one** statement that removes the shadowing so that the last line works.

In [ ]:
max = 10

try:
    print(max(3, 7))
except TypeError as e:
    print("TypeError:", e)

del max

print(max(3, 7))

# --- Check your answer (do not edit) ---
assert max(3, 7) == 7
print("Correct!")

In which scope did Python find `max` at the first `print`? What does this show about the LEGB order?

**Answer:** In the **global** scope: our `max = 10` *shadows* the built-in `max`, because G is searched before B. `del max` removes the global name, so the lookup falls through to the built-ins again.

## Part 2: Modules

#### Exercise 2.1 — Using a module

Using the `fibo` module imported above:

* **(a)** store in `fib_sum` the sum of all Fibonacci numbers strictly below 1000 (use `fibo.fib2`);
* **(b)** write `count_fib(n)`, which returns how many Fibonacci numbers are strictly less than `n`, again relying on `fibo`.

In [ ]:
# (a)
fib_sum = sum(fibo.fib2(1000))

# (b)
def count_fib(n):
    return len(fibo.fib2(n))

# --- Check your answer (do not edit) ---
assert fib_sum == 2583, f"fib_sum is {fib_sum}"
assert count_fib(100) == 12
assert count_fib(1) == 1
assert count_fib(2) == 3
print("Correct!")

**(c)** Run the cell below.

In [ ]:
print("this notebook:", __name__)
print("the fibo module:", fibo.__name__)

Why is the value of `__name__` different in the two cases?

**Answer:** A module's `__name__` is the name under which it was imported (`'fibo'`). The code of the notebook is not imported by anybody: it is the top-level environment, whose `__name__` is `'__main__'`.

#### Exercise 2.2 — The different forms of `import`

The cell below creates a small module `mymath.py` and a helper `star_import(package)` that runs `from package import *` in a scratch namespace and returns the (sorted) names that arrive. Just run it.

In [ ]:
mymath_source = '''
PI = 3.14159

def area(r):
    return PI * r * r

def perimeter(r):
    return 2 * PI * r

def _secret():
    return 42
'''

with open('mymath.py', 'w') as f:
    f.write(mymath_source)

import importlib
importlib.invalidate_caches()

def star_import(package):
    # Run `from <package> import *` in a scratch namespace; return the sorted names that arrive
    ns = {}
    exec(f"from {package} import *", ns)
    return sorted(n for n in ns if not n.startswith("__"))

* **(a)** Import the module under the alias `mm` and store in `area_2` the area of a circle of radius 2.
* **(b)** Import **only** the function `perimeter`, under the name `circ`, and store in `perim_1` the perimeter of a circle of radius 1.
  (After (a) and (b), the names `mymath` and `perimeter` must **not** exist in the notebook namespace.)

In [ ]:
# (a)
import mymath as mm
area_2 = mm.area(2)

# (b)
from mymath import perimeter as circ
perim_1 = circ(1)

# --- Check your answer (do not edit) ---
assert abs(area_2 - 12.56636) < 1e-9
assert abs(perim_1 - 6.28318) < 1e-9
assert "mymath" not in globals() and "perimeter" not in globals()
print("Correct!")

**(c)** Which names does `from mymath import *` bring in? Give your prediction as a list sorted alphabetically (is `_secret` in it?).

In [ ]:
prediction_c = ["PI", "area", "perimeter"]

# --- Check your answer (do not edit) ---
actual_c = star_import("mymath")
assert prediction_c == actual_c, "Not quite: re-read the rule about names starting with an underscore."
print("Correct!")

#### Exercise 2.3 — Executing modules as scripts

Write (as a string, then to disk) a module `greeter.py` with:

* a function `greet(name)` returning the string `"Hello, <name>!"`;
* a guard `if __name__ == "__main__":` such that running `python greeter.py Ada` prints `Hello, Ada!`, and running `python greeter.py` (no argument) prints `Hello, world!`.

Importing the module must **not** print anything.

In [ ]:
greeter_source = '''
import sys

def greet(name):
    return f"Hello, {name}!"

if __name__ == "__main__":
    name = sys.argv[1] if len(sys.argv) > 1 else "world"
    print(greet(name))
'''

with open('greeter.py', 'w') as f:
    f.write(greeter_source)

# --- Check your answer (do not edit) ---
import contextlib, importlib, io, subprocess, sys
importlib.invalidate_caches()
sys.modules.pop("greeter", None)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    import greeter
assert buf.getvalue() == "", "importing the module must not print anything"
assert greeter.greet("Ada") == "Hello, Ada!"

out = subprocess.run([sys.executable, "greeter.py", "Ada"], capture_output=True, text=True).stdout.strip()
assert out == "Hello, Ada!", repr(out)
out = subprocess.run([sys.executable, "greeter.py"], capture_output=True, text=True).stdout.strip()
assert out == "Hello, world!", repr(out)
print("Correct!")

#### Exercise 2.4 — The module search path

The cell below creates a directory `extra_modules/` containing a module `secret_mod.py`. Complete it:

* **(a)** try `import secret_mod`; it must fail. Catch the exception and store its class name in `err_name`;
* **(b)** make the import succeed **without moving the file** (hint: `sys.path`), then import it;
* **(c)** remove `extra_dir` from `sys.path` again.

In [ ]:
import os, sys, importlib

extra_dir = os.path.abspath("extra_modules")
os.makedirs(extra_dir, exist_ok=True)
with open(os.path.join(extra_dir, "secret_mod.py"), "w") as f:
    f.write('MESSAGE = "found me"\n')
sys.modules.pop("secret_mod", None)     # (so that the cell can be re-run)

# (a)
err_name = None
try:
    import secret_mod
except ImportError as e:
    err_name = type(e).__name__

# (b)
sys.path.append(extra_dir)
import secret_mod

# (c)
sys.path.remove(extra_dir)

# --- Check your answer (do not edit) ---
assert err_name == "ModuleNotFoundError", err_name
assert secret_mod.MESSAGE == "found me"
assert extra_dir not in sys.path
print("Correct!")

What is the difference between `sys.path.append(d)` and `sys.path.insert(0, d)`? When could it matter?

**Answer:** The list is searched in order, so it is a matter of **priority**. `insert(0, d)` puts `d` first: a module found there hides any module of the same name located later in the path (even a standard one: a local file `random.py` would shadow the standard `random`). `append(d)` gives `d` the lowest priority: it is used only if the name is not found anywhere else.

#### Exercise 2.5 — Compiled files and reloading

**(a)** Ask the standard library (look at `importlib.util.cache_from_source`) where the compiled version of `fibo.py` is stored, and store the result in `pyc_path`. Do not hard-code the Python version.

In [ ]:
import importlib.util, os

pyc_path = importlib.util.cache_from_source("fibo.py")
print(pyc_path)

# --- Check your answer (do not edit) ---
assert os.path.abspath(pyc_path) == os.path.abspath(fibo.__cached__)
print("Correct!")

**(b)** What does the `cpython-3XX` part of the file name tell you, and why is it useful?

**Answer:** It is a tag identifying the interpreter and its version. Different Python versions use different bytecode; thanks to the tag, several versions can share the same source directory without overwriting each other's cached files.

**(c)** A module is executed only the first time it is imported. In the cell below:

1. the module `counter_mod` is imported, then its file changes on disk;
2. a second `import counter_mod` is executed: what does `VALUE` contain afterwards?
3. **TODO**: force Python to re-execute the module so that `VALUE` becomes `22`.

In [ ]:
import importlib, sys

sys.modules.pop("counter_mod", None)     # (so that the cell can be re-run)
with open("counter_mod.py", "w") as f:
    f.write("VALUE = 1\n")
importlib.invalidate_caches()

import counter_mod
print("first import:", counter_mod.VALUE)

# The file changes on disk. (The new text has a different length on purpose: Python decides whether a cached
# .pyc file is stale from the source's modification time, in whole seconds, *and* its size.)
with open("counter_mod.py", "w") as f:
    f.write("VALUE = 22\n")

import counter_mod                       # a second import...
value_after_import = counter_mod.VALUE

importlib.reload(counter_mod)
value_after_reload = counter_mod.VALUE

# --- Check your answer (do not edit) ---
assert value_after_import == 1, "a second `import` must NOT re-run the module"
assert value_after_reload == 22
print("Correct!")

#### Exercise 2.4 — Standard modules

**(a)** Some modules are compiled into the interpreter itself (like `sys`), others are loaded from a file. Using `sys.builtin_module_names`, fill in the dictionary (`True` = built-in module).

**(b)** Using only standard modules (hint: `re` and `collections.Counter`), write `top_word(text)` which returns the most frequent word of `text`, in lower case, ignoring punctuation.

In [ ]:
import sys, json
import re
from collections import Counter

# (a)
is_builtin = {
    "sys": "sys" in sys.builtin_module_names,
    "fibo": "fibo" in sys.builtin_module_names,
    "json": "json" in sys.builtin_module_names,
}

# (b)
def top_word(text):
    words = re.findall(r"[a-z']+", text.lower())
    return Counter(words).most_common(1)[0][0]

# --- Check your answer (do not edit) ---
assert is_builtin == {"sys": True, "fibo": False, "json": False}, is_builtin
assert top_word("The cat and the hat. The end!") == "the"
assert top_word("Red fish, blue fish, red fish!") == "fish"
print("Correct!")

## Part 3: Packages

#### Exercise 3.1 — Building a package on disk

First, a small helper used by several exercises of this part: `forget(prefix)` removes a package and its submodules from `sys.modules` (it does what the loops in the cells below do), so that the next `import` starts from scratch. Just run it.

In [ ]:
import os, sys, importlib

def forget(prefix):
    # Remove `prefix` and all its submodules from sys.modules, forcing a fresh import next time
    for name in [m for m in sys.modules if m == prefix or m.startswith(prefix + ".")]:
        del sys.modules[name]
    importlib.invalidate_caches()

**(a)** Add a new subpackage `sound/analysis` to the package built above. It must contain:

* an `__init__.py` (a one-line docstring is enough);
* a module `spectrum.py` defining `peak_frequency(data)`, which simply returns `440`.

In [ ]:
os.makedirs("sound/analysis", exist_ok=True)
with open("sound/analysis/__init__.py", "w") as f:
    f.write("'''Subpackage for sound analysis.'''\n")
with open("sound/analysis/spectrum.py", "w") as f:
    f.write("def peak_frequency(data):\n    return 440\n")

# --- Check your answer (do not edit) ---
assert os.path.isfile("sound/analysis/__init__.py")
assert os.path.isfile("sound/analysis/spectrum.py")
assert "def peak_frequency" in open("sound/analysis/spectrum.py").read()
print("Correct!")

**(b)** The next cell creates a directory `nsdemo/` with a module `utils.py` but **deliberately without an `__init__.py`**.
Predict whether `import nsdemo.utils` works (`True`/`False`), then run the cell.

In [ ]:
prediction_works = True

os.makedirs("nsdemo", exist_ok=True)
with open("nsdemo/utils.py", "w") as f:
    f.write("def hello():\n    return 'hi'\n")
importlib.invalidate_caches()

try:
    import nsdemo.utils
    works = True
except ImportError:
    works = False
print("works:", works)
if works:
    print("nsdemo.__file__ :", getattr(nsdemo, "__file__", None))
    print("nsdemo.__path__ :", nsdemo.__path__)

# --- Check your answer (do not edit) ---
assert prediction_works == works
print("Correct!")

`nsdemo` was found although it has no `__init__.py`. What kind of package is it? Given this, why do we still put an `__init__.py` in our packages?

**Answer:** Since Python 3.3 a directory without `__init__.py` can still be imported: it is a **namespace package** (it has no `__file__`, and its `__path__` is a special `_NamespacePath`). Namespace packages are meant for splitting one package over several directories. A **regular** package, with an `__init__.py`, can run initialization code, define `__all__`, and re-export names; that is why we keep the file in ordinary packages.

#### Exercise 3.2 — Importing from a package

Use the `sound` package to complete each item **with the import style requested**:

* **(a)** with the full dotted name: `import sound.effects.echo`, then call `echofilter(b"x")`;
* **(b)** with `from sound.effects import ...`, binding the submodule to the name `echo_mod`, then call its `echofilter(b"x")`;
* **(c)** import `read` from `sound.formats.wavread` under the name `read_wav`, and read `"song.wav"`;
* **(d)** import the module `spectrum` of the subpackage created in Exercise 3.1 in the style you prefer, and call `peak_frequency(b"x")`.

In [ ]:
# (a)
import sound.effects.echo
out_a = sound.effects.echo.echofilter(b"x")

# (b)
from sound.effects import echo as echo_mod
out_b = echo_mod.echofilter(b"x")

# (c)
from sound.formats.wavread import read as read_wav
out_c = read_wav("song.wav")

# (d)
from sound.analysis import spectrum
out_d = spectrum.peak_frequency(b"x")

# --- Check your answer (do not edit) ---
assert out_a == b"x" and out_b == b"x"
assert out_c == b"fake-wav-data"
assert out_d == 440
print("Correct!")

**(e)** For each statement below, predict the outcome: `"OK"`, `"ImportError"` or `"ModuleNotFoundError"`
(the exact class name; `ModuleNotFoundError` is a subclass of `ImportError`). Re-read the note above about `from package import item` versus `import item.subitem`.

In [ ]:
prediction_e = {
    "import sound.effects.echo.echofilter": "ModuleNotFoundError",
    "from sound.effects.echo import echofilter": "OK",
    "from sound.effects import nonexistent": "ImportError",
}

# --- Check your answer (do not edit) ---
actual_e = {}
for stmt in prediction_e:
    try:
        exec(stmt, {})
        actual_e[stmt] = "OK"
    except Exception as e:
        actual_e[stmt] = type(e).__name__
assert prediction_e == actual_e, "Not quite: in `import a.b.c` every item but the last must be a package."
print("Correct!")

#### Exercise 3.3 — `from package import *` and `__all__`

The cell below creates a package `toolbox` with three modules `a`, `b`, `c` (each defining `NAME`) and an **empty** `__init__.py`.

In [ ]:
os.makedirs("toolbox", exist_ok=True)
for name in ["a", "b", "c"]:
    with open(f"toolbox/{name}.py", "w") as f:
        f.write(f'NAME = "{name}"\n')
with open("toolbox/__init__.py", "w") as f:
    f.write("")

def star_import(package):
    # Run `from <package> import *` in a scratch namespace; return the sorted names that arrive
    ns = {}
    exec(f"from {package} import *", ns)
    return sorted(n for n in ns if not n.startswith("__"))

**(a)** Predict the list of names brought in by `from toolbox import *` right now (nothing has imported `a`, `b` or `c` yet).

In [ ]:
prediction_a = []

# --- Check your answer (do not edit) ---
assert prediction_a == star_import("toolbox"), "Not quite: without __all__, submodules are not imported by *"
print("Correct!")

**(b)** Rewrite `toolbox/__init__.py` so that `from toolbox import *` brings in only `a` and `b`, not `c`.

In [ ]:
with open("toolbox/__init__.py", "w") as f:
    f.write('__all__ = ["a", "b"]\n')

forget("toolbox")
result_b = star_import("toolbox")
print(result_b)

# --- Check your answer (do not edit) ---
assert result_b == ["a", "b"]
print("Correct!")

**(c)** `c` is not in `__all__`. Can you still import it explicitly? Do it, and store `c.NAME` in `c_name`.

In [ ]:
from toolbox import c as toolbox_c
c_name = toolbox_c.NAME

# --- Check your answer (do not edit) ---
assert c_name == "c"
print("Correct!")

**Teaching notes.** `__all__` only controls what `import *` brings in; it does not prevent explicit imports.

#### Exercise 3.4 — References inside a package

**(a)** Create the module `sound/analysis/report.py` using **relative imports** only:

* it imports its sibling module `spectrum`;
* it imports `equalizer` from the sibling subpackage `filters`;
* it defines `analyze(data)`, which equalizes `data` (default preset) and returns the peak frequency of the result;
* when run as the main program it prints `analyze(b"demo")`.

In [ ]:
report_source = '''
from . import spectrum
from ..filters import equalizer

def analyze(data):
    equalized = equalizer.equalize(data)
    return spectrum.peak_frequency(equalized)

if __name__ == "__main__":
    print(analyze(b"demo"))
'''

with open("sound/analysis/report.py", "w") as f:
    f.write(report_source)

# --- Check your answer (do not edit) ---
forget("sound")
from sound.analysis.report import analyze
assert analyze(b"demo") == 440
assert "from ." in report_source, "use RELATIVE imports here"
print("Correct!")

**(b)** Now run this module in two different ways, and look at the results.

In [ ]:
import subprocess

# 1) as a plain script, given its file path
direct = subprocess.run([sys.executable, "sound/analysis/report.py"], capture_output=True, text=True)
print("direct  -> exit code", direct.returncode, "|", direct.stderr.strip().splitlines()[-1])

# 2) as a module of the package
as_module = subprocess.run([sys.executable, "-m", "sound.analysis.report"], capture_output=True, text=True)
print("with -m -> exit code", as_module.returncode, "| last line:", as_module.stdout.strip().splitlines()[-1])

Why does the first way fail while the second one works?

**Answer:** Run by path, the file is executed as a top-level script: `__name__` is `'__main__'` and there is **no parent package**, so `from . import spectrum` has nothing to be relative to (`ImportError: attempted relative import with no known parent package`). With `-m sound.analysis.report`, Python knows the module belongs to the package `sound.analysis`, so the relative imports can be resolved.

**(c)** Create `sound/analysis/report_abs.py`: the same module, but using **absolute imports** (`sound.analysis...`, `sound.filters...`).

In [ ]:
report_abs_source = '''
from sound.analysis import spectrum
from sound.filters import equalizer

def analyze(data):
    return spectrum.peak_frequency(equalizer.equalize(data))
'''

with open("sound/analysis/report_abs.py", "w") as f:
    f.write(report_abs_source)

# --- Check your answer (do not edit) ---
forget("sound")
from sound.analysis.report_abs import analyze as analyze_abs
assert analyze_abs(b"demo") == 440
assert "from sound" in report_abs_source and "from ." not in report_abs_source
print("Correct!")

#### Exercise 3.5 — The `__path__` attribute

**(a)** Store in `effects_dirs` the list of directories in which the package `sound.effects` looks for its submodules.

**(b)** The cell creates a directory `extra_effects/`, *outside* the package, containing a module `reverse.py`.
Extend `sound.effects.__path__` so that `from sound.effects import reverse` works.

In [ ]:
import os, importlib
import sound.effects

# (a)
effects_dirs = list(sound.effects.__path__)

# (b)
extra_effects = os.path.abspath("extra_effects")
os.makedirs(extra_effects, exist_ok=True)
with open(os.path.join(extra_effects, "reverse.py"), "w") as f:
    f.write("def reverse(data):\n    return data[::-1]\n")
importlib.invalidate_caches()

sound.effects.__path__.append(extra_effects)

from sound.effects import reverse

# --- Check your answer (do not edit) ---
assert effects_dirs[0].endswith(os.path.join("sound", "effects"))
assert reverse.reverse(b"abc") == b"cba"
print("Correct!")

#### Exercise 3.6 — Another submodule in `__all__`

1. Add a `filters/karaoke.py` module with a function `remove_vocals(data)` that just prints a message and returns `data`.
2. Give `sound/filters/__init__.py` an `__all__` list containing `"equalizer"` and `"karaoke"`.
3. Import everything from `sound.filters` with `from sound.filters import *` and call `karaoke.remove_vocals(...)`.

In [ ]:
with open('sound/filters/karaoke.py', 'w') as f:
    f.write(
        'def remove_vocals(data):\n'
        '    print(f"[karaoke] removing vocals from {data!r}")\n'
        '    return data\n'
    )

with open('sound/filters/__init__.py', 'w') as f:
    f.write(
        "'''Subpackage for filters.'''\n"
        '__all__ = ["equalizer", "karaoke"]\n'
    )

forget("sound")
from sound.filters import *
karaoke.remove_vocals(b"raw-audio")

# --- Check your answer (do not edit) ---
assert karaoke.remove_vocals(b"raw") == b"raw"
assert star_import("sound.filters") == ["equalizer", "karaoke"]
print("Correct!")

#### Exercise 3.7 — Challenge: your own package

Build a package `textkit` from scratch, with this structure:

```
textkit/
    __init__.py    re-exports normalize and word_count (relative imports) and defines __all__
    cleaning.py    normalize(text): lower-case, and collapse all whitespace runs into single spaces (no leading/trailing space)
    stats.py       word_count(text): number of words of normalize(text); uses normalize through a RELATIVE import
```

so that `from textkit import word_count` works and `from textkit import *` brings in exactly `normalize` and `word_count`.

In [ ]:
os.makedirs("textkit", exist_ok=True)

files = {
    "textkit/cleaning.py": '''
def normalize(text):
    return " ".join(text.lower().split())
''',
    "textkit/stats.py": '''
from .cleaning import normalize

def word_count(text):
    return len(normalize(text).split())
''',
    "textkit/__init__.py": '''
from .cleaning import normalize
from .stats import word_count

__all__ = ["normalize", "word_count"]
''',
}

for path, content in files.items():
    with open(path, "w") as f:
        f.write(content)

# --- Check your answer (do not edit) ---
forget("textkit")
from textkit import normalize, word_count
assert normalize("  Hello   WORLD \n") == "hello world"
assert word_count("  Hello   hello  World ") == 3
assert word_count("") == 0
import textkit
assert sorted(textkit.__all__) == ["normalize", "word_count"]
assert star_import("textkit") == ["normalize", "word_count"]
print("Correct!")

### Cleaning up

Removes every file and directory created by the exercises.

In [ ]:
import shutil, os

for f in ['fibo.py', 'mymath.py', 'greeter.py', 'counter_mod.py']:
    if os.path.exists(f):
        os.remove(f)

for d in ['sound', 'extra_modules', 'nsdemo', 'toolbox', 'extra_effects', 'textkit', '__pycache__']:
    if os.path.exists(d):
        shutil.rmtree(d)

print("Cleaned up.")